# 01 - Data analysis: AI-generated Shorts in child feeds

> **Demo-data notice.** `data/raw/test_run.jsonl` currently contains a small **synthetic** log
> (every record has `"synthetic": true`). It only shows that the pipeline works end to end.
> Replace the file with the real logs (same schema, see `src/make_demo_data.py`) before drawing any conclusions.

This notebook answers the three research questions of the project:

| RQ | Question | Effect we report |
|----|----------|------------------|
| RQ1 | Is the AI share higher in child feeds than in adult feeds? | difference in AI share (percentage points) |
| RQ2 | Does watching AI videos increase the AI share more than watching human videos? | difference in the change of AI share (pp) |
| RQ3 | Does a multimodal detector beat a metadata-only baseline? | gain in macro-F1 |

**Structure:** (1) data ingestion and warm-up filtering, (2) statistical aggregation, (3) visualization and export.
All random steps use seed 42. To reproduce: *Kernel -> Restart & Run All*.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)  # the bootstraps inside src/process_data.py also use seed 42

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import process_data as pdm   # data processing and statistics
import visualize as viz      # figures

pd.set_option("display.width", 160)
print("repo root folder:", ROOT.name)

ModuleNotFoundError: No module named 'pandas'

## 1. Data ingestion and warm-up filtering

A new account has no watch history, so its very first session is a **cold start**: the feed is unstable and not
comparable with later sessions. These sessions are stored in the raw log with `phase = "warmup"` and are
**discarded**; only `phase = "eval"` sessions are analysed.

In [ ]:
feeds, videos = pdm.load_raw(ROOT / "data" / "raw" / "test_run.jsonl")
feeds = pdm.add_share(feeds)
print(f"feed sessions: {len(feeds)}   hand-labelled videos: {len(videos)}")
display(feeds.groupby(["phase", "profile_type"]).size().unstack(fill_value=0))

In [ ]:
sessions, n_dropped = pdm.drop_warmup(feeds)
print(f"dropped {n_dropped} cold-start warm-up sessions, kept {len(sessions)} evaluation sessions")
display(sessions[["session_id", "profile_type", "arm", "session_idx", "n_ai", "n_unavailable", "ai_share"]].head())

## 2. Statistical aggregation

* **Unit of analysis:** the *session* for RQ1 and the *account* for RQ2 (videos inside one feed are not independent).
* **Dispersion:** mean +- SD, median, p25, p75, p95, p99.
* **Uncertainty:** 95 % percentile bootstrap CI (10 000 resamples, seed 42).
* The **effect size with its CI is the primary result**; the p-value of the one-sided Welch t-test is secondary.

In [ ]:
r1 = pdm.rq1(sessions)
display(pd.DataFrame({g: pdm.describe(r1[g] * 100) for g in ["adult", "child"]}).T.round(2)
        .rename_axis("RQ1 profile (AI share, %)"))
lo, hi = (v * 100 for v in r1["ci"])
print(f"RQ1 effect (child - adult): {r1['diff'] * 100:.1f} pp, 95% bootstrap CI [{lo:.1f}, {hi:.1f}]")
print(f"secondary: Welch t = {r1['t']:.2f}, one-sided p = {r1['p']:.2g} (session level)")

In [ ]:
r2 = pdm.rq2(sessions)
display(pd.DataFrame({a: pdm.describe(x * 100) for a, x in r2["groups"].items()}).T.round(2)
        .rename_axis("RQ2 arm (change in AI share, pp)"))
lo, hi = (v * 100 for v in r2["ci"])
print(f"RQ2 effect (AI-dwell - human-dwell): {r2['diff'] * 100:.1f} pp, 95% bootstrap CI [{lo:.1f}, {hi:.1f}]")
print(f"secondary: Welch t = {r2['t']:.2f}, one-sided p = {r2['p']:.2g} (account level); no-dwell is an extra control")

In [ ]:
r3 = pdm.rq3(videos)
table = pd.DataFrame([
    {"feature set": k, **{m: v[m] for m in ["macro_f1", "macro_precision", "macro_recall"]},
     "ci_low": v["ci"][0], "ci_high": v["ci"][1]}
    for k, v in r3["sets"].items()]).set_index("feature set")
display(table.round(3))
print(f"train videos: {r3['n_train']}, test videos: {r3['n_test']}")
print(f"RQ3 gain (multimodal - metadata-only): {r3['gain']:.3f}, 95% bootstrap CI "
      f"[{r3['gain_ci'][0]:.3f}, {r3['gain_ci'][1]:.3f}]; threshold 0.05 reached: {r3['gain_reaches_threshold']}")

In [ ]:
guards = pd.DataFrame(pdm.guardrails(sessions, videos))
display(guards.round(3))

## 3. Visualization and export

Every figure has labelled axes with units, a 95 % bootstrap CI, a colour-blind-safe palette with different
markers / line styles / hatches, and is saved as vector `.pdf` and `.png` (300 DPI) in `reports/figures/`.

In [ ]:
summary, res = pdm.build_summary()
fig1 = viz.make_fig1(res["trajectory"])
viz.save_figure(fig1, "fig1_ai_share_vs_session")
plt.show()

In [ ]:
fig2 = viz.make_fig2(res["rq1"])
viz.save_figure(fig2, "fig2_distribution_boxplot")
plt.show()

In [ ]:
fig3 = viz.make_fig3(res["rq3"])
viz.save_figure(fig3, "fig3_f1_by_feature_set")
plt.show()

In [ ]:
out = pdm.save_summary(summary)
print("saved", out.relative_to(ROOT))
display(summary[summary["metric"].str.contains("diff|gain")][["rq", "metric", "value", "ci_low", "ci_high", "note"]].round(3))

## 4. How to read the results

* RQ1, RQ2 and RQ3 are judged by the **effect size and its 95 % CI**, not by the p-value alone.
* With few sessions, accounts and test videos the CIs are wide; more real data will narrow them.
* The numbers above come from **synthetic demo data** and say nothing about YouTube yet.